# GT-Centric Cell Tracking Viewer & Exporter Pipeline (S2.3)

このノートブックは、**Ground Truth (GT) を100%基準**とした細胞トラッキング可視化データ (`viewer_data.json`) を抽出・生成し、3軸 MIP 射影画像と共に GitHub Pages (`kito2718/kaggle_Biohub-Cell_Tracking_During_Development2` の `gh-pages` ブランチ) へ自動デプロイするための統合処理ノートブックです。

---

## プロジェクト構成

本ノートブックは Kaggle Notebook環境専用の構成で動作します。

### Kaggle Notebook環境の構成
```text
/kaggle/
├── working/                                    # 作業ディレクトリ (カレントディレクトリ)
│   └── s2_03_gt_html_viewer.ipynb              # 本実行ノートブック
└── input/
    ├── competitions/
    │   └── biohub-cell-tracking-during-development/ # コンペ公式データセット
    │       ├── train/                         # 訓練用データセット (.zarr / .geff)
    │       │   ├── xxxx.zarr/
    │       │   └── xxxx.geff/
    │       └── test/                          # 提出用データセット (.zarr)
    └── datasets/
        └── aaaa1597/
            ├── zarr-offline-installation-wheels/ # オフラインインストール用 zarr Wheels
            ├── tracksdata-wheels/                # オフラインインストール用 tracksdata Wheels (*.whl)
            ├── kaggle-cell-tracking-competition/  # 評価・処理用ソースコード (src/)
            └── btc-s106-progress/             # 継続実行・途中再開(Resume)用Dataset
```

## コミット対象構造 (`gh-pages` ブランチ)
```text
https://github.com/kito2718/kaggle_Biohub-Cell_Tracking_During_Development2 (branch: gh-pages)
 ├ index.html
 └ viewer_data/
 　 ├ datasets.json
 　 ├ 44b6_f28707c6/
 　 │ ├ mips/
 　 │ │ ├ frame_000_xy.png
 　 │ │ ├ frame_000_xz.png
 　 │ │ ├ frame_000_yz.png
 　 │ │ └ ...
 　 │ └ viewer_data.json
 　 └ 44b6_12dfb391/ ...
```
### 必要な Kaggle Datasets & Add-ons (Secrets) の事前準備手順

本ノートブックを Kaggle 環境で安定して連続実行・自動デプロイするために、以下の Kaggle Datasets および Secrets の設定を行ってください。

1. **必要な Kaggle Datasets の追加 (+ Add Data)**:
   - **`zarr-offline-installation-wheels`** (`/kaggle/input/datasets/aaaa1597/zarr-offline-installation-wheels`):
     - インターネット接続オフの環境で `zarr` をインストールするためのオフライン Wheel 群データセット。
   - **`tracksdata-wheels`** (`/kaggle/input/datasets/aaaa1597/tracksdata-wheels`):
     - インターネット接続オフの環境で `tracksdata`, `geff`, `btrack` 等をインストールするためのオフライン Wheel 群データセット (`*.whl`)。
   - **`kaggle-cell-tracking-competition`** (`/kaggle/input/datasets/aaaa1597/kaggle-cell-tracking-competition`):
     - 公式評価指標および `tracking_cellmot` / `tracksdata` モジュール群が含まれるソースコードデータセット (`src/`)。
   - **`btc-s106-progress`** (`/kaggle/input/datasets/aaaa1597/btc-s106-progress`):
     - 9時間セッション制限対策の継続実行・途中再開 (Resume) 用 Dataset (`progress.json` や過去のチェックポイントを保持)。

2. **Add-ons > Secrets の設定**:
   - `GITHUB_TOKEN`: GitHub への可視化データ自動同期・プッシュに必要な GitHub Personal Access Token。
   - `KAGGLE_USERNAME`: Kaggle ユーザー名 (`aaaa1597`)。
   - `KAGGLE_KEY`: Kaggle API Token Key。

---

## 処理フローチャート (Pipeline Flowchart)

パイプライン全体の一括処理、途中再開 (Resume) 判定、GT 100%ノード抽出 (degree=0 の孤立ノードを含む)、および GitHub Pages への `--depth 1` 高速デプロイの処理フローです。枠内の名称はノートブックに実装されている実際の関数名およびセル番号と対応しています。

```mermaid
graph TD
    classDef default fill:#f9f9f9,stroke:#333,stroke-width:1px;
    classDef loop fill:#e1f5fe,stroke:#0288d1,stroke-width:2px;
    classDef cell fill:#f3e5f5,stroke:#8e24aa,stroke-width:2px;
    classDef func fill:#efebe9,stroke:#5d4037,stroke-width:1px;
    classDef cond fill:#fff9c4,stroke:#fbc02d,stroke-width:1px;

    Start([処理開始]) --> Cell2["Cell 2: オフライン パッケージライブラリインストール<br>(zarr, tracksdata, btrack, geff)"]
    Cell2 --> Cell3["Cell 3: 環境パラメータ設定 & モジュールインポート"]
    Cell3 --> Cell4["Cell 4: check_environment()<br>(動作環境 Fail-Fast チェック)"]
    class Cell4 func;
    Cell4 --> Cell6["Cell 6: get_dataset_pairs()<br>(.zarr & .geff ペア探索)"]
    class Cell6 func;

    Cell6 --> Cell12_Init["Cell 12: process_all_datasets()<br>(全データセットバッチ処理開始)"]
    class Cell12_Init func;

    subgraph Cell12_Loop ["Cell 12: 全データセットバッチ処理ループ"]
        LoopStart{"データセットループ開始"}
        class LoopStart loop;

        LoopStart --> CheckSkip{"CONTINUOUS_FLAG == True <br>&& 処理完了済みデータセット?"}
        class CheckSkip cond;

        CheckSkip -- Yes (スキップ) --> LoopEndDummy[ ]
        style LoopEndDummy fill:none,stroke:none,width:0px,height:0px;

        CheckSkip -- No --> StepGT["Cell 7: export_gt_viewer_data()<br>1. GT 100%全載せ抽出<br>2. degree=0 孤立GTノードカウント<br>3. TP/FP/FN & グローバル統計算出"]
        class StepGT func;

        StepGT --> StepMIP["Cell 9: generate_pred_graph() & ensure_mip_images()<br>3軸 MIP 射影画像 (xy/xz/yz) の配置・生成"]
        class StepMIP func;

        StepMIP --> SaveCheck["Cell 11: save_completed_dataset()<br>進捗チェックポイントの更新"]
        class SaveCheck func;

        SaveCheck --> LoopEndDummy
    end

    Cell12_Loop --> Cell14["Cell 14: ensure_index_html()<br>HTMLビューアー テンプレート生成"]
    class Cell14 func;

    Cell14 --> Cell15["Cell 15: push_to_github_pages()<br>1. git clone --branch gh-pages --depth 1 (高速浅いクローン)<br>2. index.html, viewer_data.json, mips/, datasets.json 同期<br>3. remote repo へ自動 push"]
    class Cell15 func;

    Cell15 --> Cell16["Cell 16: メインエントリーポイント実行"]
    class Cell16 func;

    Cell16 --> End([処理完了])
```


In [ ]:
# Cell 2: オフライン パッケージライブラリインストール
!pip install --no-index --find-links=/kaggle/input/datasets/aaaa1597/zarr-offline-installation-wheels/zarr_wheels zarr
!pip install --no-index --find-links=/kaggle/input/datasets/aaaa1597/tracksdata-wheels rustworkx bidict ilpy imagecodecs polars btrack zarr Pillow
!pip install --no-index --no-deps --find-links=/kaggle/input/datasets/aaaa1597/tracksdata-wheels geff geff-spec
!pip install --no-index --no-deps --find-links=/kaggle/input/datasets/aaaa1597/tracksdata-wheels tracksdata


In [ ]:
# Cell 3: 環境・実行パラメータ設定 & モジュールインポート (CONFIGURATION & IMPORTS)
import os
import sys
import glob
import time
import json
import shutil
import tempfile
import numpy as np
import polars as pl
if not hasattr(pl, 'Float16'):
    pl.Float16 = pl.Float32
import zarr
from pathlib import Path
from PIL import Image

# ==============================================================================
# Cell 3: 環境・実行パラメータ設定 & モジュールインポート (CONFIGURATION & IMPORTS)
# ==============================================================================
# 1. 途中再開 (Resume) & チェックポイント設定
CONTINUOUS_FLAG = True       # True: 自動チェックポイント保存 & スキップを有効化
RESET_CHECKPOINT = False     # True: 過去のチェックポイントを一度クリアして一からスタート

# 2. index.htmlのみ作成,RAW CAVのみ作成フラグ
ONLY_CREATE_INDEX_HTML = False # True: 重い細胞検出計算をスキップし、index.html 生成 → GitHub デプロイのみを高速実行
ONLY_CREATE_RAW_CSV = False    # True: 重い細胞検出計算をスキップし、統合 RAW CSV 抽出 → GitHub デプロイのみを高速実行

# 3. GitHub Pages 自動デプロイ設定
PUSH_TO_GITHUB = True         # True: 処理結果を GitHub Pages へ自動反映
GITHUB_REPO = 'https://github.com/kito2718/kaggle_Biohub-Cell_Tracking_During_Development2.git'
BRANCH_NAME = 'gh-pages'
from kaggle_secrets import UserSecretsClient
GITHUB_TOKEN = UserSecretsClient().get_secret("GITHUB_TOKEN") if PUSH_TO_GITHUB else ''

# 4. データセットパス設定
DATASET_SLUG = "btc-s106-progress"
DATA_DIR = Path('/kaggle/input/competitions/biohub-cell-tracking-during-development/train')
WORKING_DIR = Path('/kaggle/working')

# 途中保存用 Progress Dataset ディレクトリの存在チェック
CHECKPOINT_DATASET_DIR = Path(f'/kaggle/input/datasets/aaaa1597/{DATASET_SLUG}')
if not CHECKPOINT_DATASET_DIR.exists():
    raise FileNotFoundError(f"Checkpoint dataset directory not found: {CHECKPOINT_DATASET_DIR}")

CHECKPOINT_DATASET_PATH = CHECKPOINT_DATASET_DIR / 'gt_viewer_data.json'

# 5. パス設定 & モジュールインポート
KAGGLE_SRC_DIR = '/kaggle/input/datasets/aaaa1597/kaggle-cell-tracking-competition/src'
if not os.path.exists(KAGGLE_SRC_DIR):
    raise FileNotFoundError(f"Required Kaggle source directory not found: {KAGGLE_SRC_DIR}")

if KAGGLE_SRC_DIR not in sys.path:
    sys.path.insert(0, KAGGLE_SRC_DIR)

# Ground Truth Evaluation & Metrics imports
import geff
import tracksdata as td
from tracksdata.graph import IndexedRXGraph
import tracking_cellmot.io
from tracking_cellmot.metrics import evaluate
from tracking_cellmot.io import open_dataset

print(f"Pipeline parameters initialized. CONTINUOUS_FLAG={CONTINUOUS_FLAG}, RESET_CHECKPOINT={RESET_CHECKPOINT}")
print("All required tracking_cellmot & tracksdata modules imported successfully.")

# === GPUなし環境での RuntimeError 対策 (CPUモンキーパッチ) ===
import torch
if not torch.cuda.is_available():
    def patched_process_on_gpu(
        image, tracks, scale, device,
        resample=False, target_scale=None,
        normalize=True, gamma=1.0,
        q_min=0.01, q_max=0.99, subsample_factor=1000,
        precomputed_quantiles=None
    ):
        torch_device = torch.device(device)
        image = image.astype(np.float32, copy=False)
        q1, q2 = None, None
        if normalize:
            q1 = tracking_cellmot.io._lookup_precomputed_quantile(precomputed_quantiles, q_min)
            q2 = tracking_cellmot.io._lookup_precomputed_quantile(precomputed_quantiles, q_max)
            if q1 is None or q2 is None:
                flat = image.ravel()[::subsample_factor]
                q1, q2 = np.quantile(flat, [q_min, q_max]).astype(np.float32)
            else:
                q1 = np.float32(q1)
                q2 = np.float32(q2)
        tensor = torch.from_numpy(image)
        tensor = tensor.to(torch_device, non_blocking=True)
        if normalize:
            tensor = (tensor - float(q1)) / (float(q2) - float(q1) + 1e-6)
            tensor = tensor.clamp(min=0.0)
            if gamma != 1.0:
                tensor = tensor.pow(gamma)
            tensor = tensor.clamp(0.0, 4.0)
        if resample:
            scale_arr = np.array(scale)
            target_scale_val = scale_arr.min() if target_scale is None else np.array(target_scale)
            zoom_factors = scale_arr / target_scale_val
            new_spatial_shape = (np.array(tensor.shape[1:]) * zoom_factors).astype(int).tolist()
            tensor = tensor[:, None]
            tensor = torch.nn.functional.interpolate(
                tensor, size=new_spatial_shape, mode="trilinear", align_corners=False
            )
            tensor = tensor[:, 0]
            if tracks is not None:
                tracks = tracks.copy()
                tracks[:, 1:] = tracks[:, 1:] * zoom_factors
        else:
            zoom_factors = np.ones(3, dtype=np.float32)
        return tensor, tracks, zoom_factors

    tracking_cellmot.io._process_on_gpu = patched_process_on_gpu
    print("All required tracking_cellmot & tracksdata modules imported successfully. CPU Monkey Patch applied.")

print(f"Pipeline parameters initialized. CONTINUOUS_FLAG={CONTINUOUS_FLAG}, RESET_CHECKPOINT={RESET_CHECKPOINT}")


In [ ]:
# Cell 4: 動作環境 Fail-Fast チェック関数
def check_environment():
    """
    環境要件(ライブラリ、データディレクトリ、GitHub Secret設定等)を即座に確認するFail-Fastチェック関数。
    iterdir() を用いてファイル存在を高速スキャンします。
    """
    print("Checking environment requirements...")
    
    # 1. 必須コアライブラリ・評価ライブラリのインポートチェック
    import zarr
    import polars as pl
    import geff
    import tracksdata as td
    import tracking_cellmot.io
    from tracking_cellmot.metrics import evaluate
    from tracking_cellmot.io import open_dataset
    print("  - [OK] Required libraries (zarr, polars, geff, tracksdata, tracking_cellmot) are verified.")

    # 2. データディレクトリの存在チェック
    if not DATA_DIR.exists():
        raise FileNotFoundError(f"Data directory not found: {DATA_DIR}")
    
    # 3. Zarr データセットの存在チェック
    zarr_files = sorted([p for p in DATA_DIR.iterdir() if p.name.endswith('.zarr')])
    if not zarr_files:
        raise FileNotFoundError(f"No Zarr datasets found in {DATA_DIR}")
    print(f"  - [OK] Zarr target datasets found in '{DATA_DIR}' (Count: {len(zarr_files)}).")

    # 4. GEFF Ground Truth の存在チェック
    geff_files = sorted([p for p in DATA_DIR.iterdir() if p.name.endswith('.geff')])
    if not geff_files:
        raise FileNotFoundError(f"No GEFF ground truth files found in {DATA_DIR}")
    print(f"  - [OK] GEFF ground truth files found in '{DATA_DIR}' (Count: {len(geff_files)}).")

    # 5. GitHub 自動同期設定のチェック
    if PUSH_TO_GITHUB:
        if not GITHUB_REPO or len(GITHUB_REPO.strip()) == 0:
            raise ValueError("PUSH_TO_GITHUB is True, but GITHUB_REPO is invalid or empty.")

        try:
            from kaggle_secrets import UserSecretsClient
            token = UserSecretsClient().get_secret("GITHUB_TOKEN")
        except Exception as e_sec:
            raise ValueError(f"PUSH_TO_GITHUB is True, but failed to fetch GITHUB_TOKEN from Secrets: {e_sec}") from e_sec
        
        if not token:
            raise ValueError("PUSH_TO_GITHUB is True, but GITHUB_TOKEN is not set in Secrets or env.")
        print(f"  - [OK] GitHub Auto-Push configuration (repo: '{GITHUB_REPO}') verified.")

    # 6. チェックポイント自動同期設定のチェック (CONTINUOUS_FLAG=True の時)
    if CONTINUOUS_FLAG:
        if not DATASET_SLUG:
            raise ValueError("CONTINUOUS_FLAG is True, but DATASET_SLUG is not defined.")
        try:
            from kaggle_secrets import UserSecretsClient
            u = UserSecretsClient().get_secret("KAGGLE_USERNAME")
            k = UserSecretsClient().get_secret("KAGGLE_KEY")
            if not u or not k:
                raise ValueError("KAGGLE_USERNAME or KAGGLE_KEY in Secrets is empty.")
        except Exception as e_k:
            raise ValueError(f"CONTINUOUS_FLAG is True, but failed to fetch Kaggle API secrets (KAGGLE_USERNAME / KAGGLE_KEY): {e_k}") from e_k
        print(f"  - [OK] Dataset Checkpoint Auto-Sync (slug: '{DATASET_SLUG}', secrets: verified) is valid.")

    print("Environment check PASSED successfully!")

check_environment()


## 2. Dataset Scanning & Ground Truth Loading

In [ ]:
# Cell 6: データセット対探索関数
def get_dataset_pairs(data_dir: Path):
    """
    DATA_DIR配下の Zarr と GEFF のペアを高速走査(iterdir)して取得する関数。
    """
    pairs = []
    zarr_paths = sorted([p for p in data_dir.iterdir() if p.name.endswith('.zarr')])
    for zarr_path in zarr_paths:
        dataset_name = zarr_path.stem
        geff_path = zarr_path.with_suffix('.geff')
        if geff_path.exists():
            pairs.append((dataset_name, zarr_path, geff_path))
    return pairs

dataset_pairs = get_dataset_pairs(DATA_DIR)
print(f"Discovered {len(dataset_pairs)} dataset pairs.")


In [ ]:
# Cell 7: 全 Dataset 統合 RAW CSV エクスポート (export_raw_csv_all) & 超高速 GT 基準可視化データ抽出関数 (export_gt_viewer_data)
def export_raw_csv_all(data_dir: Path, out_dir: Path):
    """
    全 Dataset の生データ .geff から全ノード・エッジ属性テーブルを直接抽出し、
    先頭に dataset 列を付与して 2 つの統合 CSV (gt_nodes.csv, gt_edges.csv) にエクスポートする関数。
    """
    import pandas as pd
    out_csv_dir = out_dir / 'viewer_data'
    out_csv_dir.mkdir(parents=True, exist_ok=True)
    
    all_nodes_dfs = []
    all_edges_dfs = []
    
    dataset_pairs = get_dataset_pairs(data_dir)
    print(f"  - [RAW CSV 統合抽出] 全 {len(dataset_pairs)} 件のデータセットを処理中...")
    
    for name, zarr_path, geff_path in dataset_pairs:
        try:
            ds_path = os.path.join(str(data_dir), name)
            ds = open_dataset(ds_path, normalize=True, require_tracks=True, device="cpu")
            gt_graph = ds.tracks
            
            nodes_df = gt_graph.node_attrs().to_pandas()
            edges_df = gt_graph.edge_attrs().to_pandas()
            
            nodes_df.insert(0, 'dataset', name)
            edges_df.insert(0, 'dataset', name)
            
            all_nodes_dfs.append(nodes_df)
            all_edges_dfs.append(edges_df)
        except Exception as e:
            print(f"  - [WARN] {name} の RAW CSV 抽出中に例外発生: {e}")

    if all_nodes_dfs:
        merged_nodes_df = pd.concat(all_nodes_dfs, ignore_index=True)
        merged_nodes_df.to_csv(out_csv_dir / 'gt_nodes.csv', index=False)
        print(f"  - [OK] viewer_data/gt_nodes.csv 出力完了: 全 {len(merged_nodes_df)} 行")

    if all_edges_dfs:
        merged_edges_df = pd.concat(all_edges_dfs, ignore_index=True)
        merged_edges_df.to_csv(out_csv_dir / 'gt_edges.csv', index=False)
        print(f"  - [OK] viewer_data/gt_edges.csv 出力完了: 全 {len(merged_edges_df)} 行")

def export_gt_viewer_data(gt_graph: IndexedRXGraph, pred_graph: IndexedRXGraph, dataset_name: str, out_dir: Path, scale=[1.0, 1.0, 1.0], spatial_shape=None):
    """
    Ground Truth (GT) 基準の可視化データ (viewer_data.json) を抽出・出力する関数。
    100% GT 保持、孤立ノード (degree=0)、TP/FP/FN 分類および全フレーム・データセットサマリーを出力します。
    """
    import pandas as pd
    from scipy.spatial import KDTree
    from tracksdata.constants import DEFAULT_ATTR_KEYS
    from tracking_cellmot.metrics import evaluate

    scale = np.array(scale, dtype=float)
    out_dataset_dir = out_dir / 'viewer_data' / dataset_name
    out_dataset_dir.mkdir(parents=True, exist_ok=True)

    # GT / Pred データフレーム取得
    try:
        gt_nodes_df = gt_graph.node_attrs().to_pandas()
        gt_edges_df = gt_graph.edge_attrs().to_pandas()
    except Exception as e:
        raise RuntimeError(f"【GT読み込みエラー】GTグラフデータの取得に失敗しました ({dataset_name}): {e}") from e

    pred_nodes_df = pred_graph.node_attrs().to_pandas()

    # 孤立 GT ノード数算定
    connected_gt_ids = set(gt_edges_df['source_id']) | set(gt_edges_df['target_id']) if not gt_edges_df.empty else set()
    isolated_gt_count = sum(1 for nid in gt_nodes_df['node_id'] if nid not in connected_gt_ids)

    # GT 親ノードマップ作成
    gt_parent_map = {}
    if not gt_edges_df.empty:
        for row in gt_edges_df.itertuples(index=False):
            gt_parent_map[int(row.target_id)] = int(row.source_id)

    # 評価実行とノード・エッジマッピング
    id_map = {}
    matched_gt_edge_pairs = set()
    
    _ = evaluate(pred_graph, gt_graph, scale=scale)
    pred_node_attrs = pred_graph.node_attrs(attr_keys=[DEFAULT_ATTR_KEYS.NODE_ID, 't', 'z', 'y', 'x', DEFAULT_ATTR_KEYS.MATCHED_NODE_ID]).to_pandas()
    for row in pred_node_attrs.itertuples(index=False):
        pred_id = int(row.node_id)
        matched_id = getattr(row, DEFAULT_ATTR_KEYS.MATCHED_NODE_ID, None)
        if matched_id is not None and not pd.isna(matched_id) and int(matched_id) != -1:
            id_map[pred_id] = int(matched_id)

    pred_edge_attrs = pred_graph.edge_attrs(attr_keys=[DEFAULT_ATTR_KEYS.EDGE_SOURCE, DEFAULT_ATTR_KEYS.EDGE_TARGET, DEFAULT_ATTR_KEYS.MATCHED_EDGE_MASK]).to_pandas()
    for row in pred_edge_attrs.itertuples(index=False):
        if bool(getattr(row, DEFAULT_ATTR_KEYS.MATCHED_EDGE_MASK, False)):
            src_m = id_map.get(int(row.source_id))
            tgt_m = id_map.get(int(row.target_id))
            if src_m is not None and tgt_m is not None:
                matched_gt_edge_pairs.add((src_m, tgt_m))
                matched_gt_edge_pairs.add((tgt_m, src_m))

    # =========================================================================
    # ⚡️ [超高速化 Core Optimization] Node ID -> Frame t & 事前グループ辞書化 (O(1))
    # =========================================================================
    gt_node_to_t = dict(zip(gt_nodes_df['node_id'].astype(int), gt_nodes_df['t'].astype(int)))
    pred_node_to_t = dict(zip(pred_nodes_df['node_id'].astype(int), pred_nodes_df['t'].astype(int)))

    gt_nodes_by_t = {int(t_val): df_sub for t_val, df_sub in gt_nodes_df.groupby('t')}
    pred_nodes_by_t = {int(t_val): df_sub for t_val, df_sub in pred_nodes_df.groupby('t')}

    gt_edges_by_t = {}
    if not gt_edges_df.empty:
        for row in gt_edges_df.itertuples(index=False):
            u, v = int(row.source_id), int(row.target_id)
            u_t = gt_node_to_t.get(u)
            if u_t is not None:
                u_t_int = int(u_t)
                if u_t_int not in gt_edges_by_t:
                    gt_edges_by_t[u_t_int] = []
                gt_edges_by_t[u_t_int].append((u, v))

    pred_edges_by_t = {}
    pred_edges_df = pred_graph.edge_attrs().to_pandas()
    if not pred_edges_df.empty:
        for row in pred_edges_df.itertuples(index=False):
            pu, pv = int(row.source_id), int(row.target_id)
            is_tp = bool(getattr(row, DEFAULT_ATTR_KEYS.MATCHED_EDGE_MASK, False)) if hasattr(row, DEFAULT_ATTR_KEYS.MATCHED_EDGE_MASK) else False
            pu_t = pred_node_to_t.get(pu)
            if pu_t is not None:
                pu_t_int = int(pu_t)
                if pu_t_int not in pred_edges_by_t:
                    pred_edges_by_t[pu_t_int] = []
                pred_edges_by_t[pu_t_int].append((pu, pv, is_tp))

    # 総フレーム数算定
    max_t_gt = int(gt_nodes_df['t'].max())
    max_t_pred = int(pred_nodes_df['t'].max())
    n_frames = max(max_t_gt, max_t_pred) + 1

    frames_dict = {}
    global_edge_counter = 1
    
    total_gt_tp_count = 0
    total_gt_fn_count = 0
    total_pred_fp_count = 0
    total_gt_edge_tp_count = 0
    total_gt_edge_fn_count = 0
    total_pred_edge_fp_count = 0

    for t in range(n_frames):
        t_gt = gt_nodes_by_t.get(t, pd.DataFrame())
        t_pred = pred_nodes_by_t.get(t, pd.DataFrame())
        
        node_matches = {}
        if not t_pred.empty and not t_gt.empty:
            pred_coords = t_pred[['z', 'y', 'x']].values * scale
            gt_coords = t_gt[['z', 'y', 'x']].values * scale
            tree = KDTree(gt_coords)
            dists, indices = tree.query(pred_coords, distance_upper_bound=15.0)
            matched_gt_indices = set()
            for p_idx, (d, g_idx) in enumerate(zip(dists, indices)):
                if d < 15.0 and g_idx not in matched_gt_indices:
                    p_id = int(t_pred.iloc[p_idx]['node_id'])
                    g_id = int(t_gt.iloc[g_idx]['node_id'])
                    node_matches[p_id] = g_id
                    matched_gt_indices.add(g_idx)

        gt_node_tp = []
        gt_node_fn = []
        pred_node_fp = []
        rendered_gt_ids = set()

        if not t_pred.empty:
            for row in t_pred.itertuples(index=False):
                p_id = int(row.node_id)
                pz, py, px = float(row.z), float(row.y), float(row.x)
                if p_id in node_matches:
                    g_id = node_matches[p_id]
                    parent_id = gt_parent_map.get(g_id, None)
                    gt_node_tp.append([pz, py, px, g_id, p_id, parent_id])
                    rendered_gt_ids.add(g_id)
                else:
                    pred_node_fp.append([pz, py, px, p_id])

        if not t_gt.empty:
            for row in t_gt.itertuples(index=False):
                g_id = int(row.node_id)
                if g_id not in rendered_gt_ids:
                    gz, gy, gx = float(row.z), float(row.y), float(row.x)
                    parent_id = gt_parent_map.get(g_id, None)
                    gt_node_fn.append([gz, gy, gx, g_id, parent_id])

        gt_edge_tp = []
        gt_edge_fn = []
        pred_edge_fp = []

        # O(1) 事前ハッシュマップ引当てによる超高速エッジ分類
        for u, v in gt_edges_by_t.get(t, []):
            edge_entry = {"edge_id": global_edge_counter, "source_nodeid": u, "target_nodeid": v}
            global_edge_counter += 1
            if (u, v) in matched_gt_edge_pairs:
                gt_edge_tp.append(edge_entry)
            else:
                gt_edge_fn.append(edge_entry)

        for pu, pv, is_tp in pred_edges_by_t.get(t, []):
            if not is_tp:
                pred_edge_fp.append({"edge_id": global_edge_counter, "source_nodeid": pu, "target_nodeid": pv})
                global_edge_counter += 1

        # フレーム度数更新
        total_gt_tp_count += len(gt_node_tp)
        total_gt_fn_count += len(gt_node_fn)
        total_pred_fp_count += len(pred_node_fp)
        total_gt_edge_tp_count += len(gt_edge_tp)
        total_gt_edge_fn_count += len(gt_edge_fn)
        total_pred_edge_fp_count += len(pred_edge_fp)

        # フレームサマリー
        tp_n, fp_n, fn_n = len(gt_node_tp), len(pred_node_fp), len(gt_node_fn)
        tp_e, fp_e, fn_e = len(gt_edge_tp), len(pred_edge_fp), len(gt_edge_fn)
        prec_n = tp_n / (tp_n + fp_n) if (tp_n + fp_n) > 0 else 0.0
        rec_n = tp_n / (tp_n + fn_n) if (tp_n + fn_n) > 0 else 0.0
        f1_n = (2 * prec_n * rec_n) / (prec_n + rec_n) if (prec_n + rec_n) > 0 else 0.0

        prec_e = tp_e / (tp_e + fp_e) if (tp_e + fp_e) > 0 else 0.0
        rec_e = tp_e / (tp_e + fn_e) if (tp_e + fn_e) > 0 else 0.0
        f1_e = (2 * prec_e * rec_e) / (prec_e + rec_e) if (prec_e + rec_e) > 0 else 0.0

        frames_dict[str(t)] = {
            "gt_node_tp": gt_node_tp,
            "pred_node_fp": pred_node_fp,
            "gt_node_fn": gt_node_fn,
            "gt_edge_tp": gt_edge_tp,
            "pred_edge_fp": pred_edge_fp,
            "gt_edge_fn": gt_edge_fn,
            "summary": {
                "precision": round(prec_n, 4),
                "recall": round(rec_n, 4),
                "f1": round(f1_n, 4),
                "edge_tp": tp_e,
                "edge_fp": fp_e,
                "edge_fn": fn_e,
                "edge_precision": round(prec_e, 4),
                "edge_recall": round(rec_e, 4),
                "edge_f1": round(f1_e, 4)
            }
        }

    viewer_payload = {
        "dataset": dataset_name,
        "metadata": {
            "num_frames": n_frames,
            "scale": scale.tolist(),
            "shape": list(spatial_shape) if spatial_shape is not None else [64, 256, 256],
            "estimated_number_of_nodes": len(gt_nodes_df)
        },
        "summary": {
            "total_gt_nodes": len(gt_nodes_df),
            "isolated_gt_nodes_count": isolated_gt_count,
            "gt_nodes_tp_count": total_gt_tp_count,
            "gt_nodes_fn_count": total_gt_fn_count,
            "pred_nodes_fp_count": total_pred_fp_count,
            "gt_edges_tp_count": total_gt_edge_tp_count,
            "gt_edges_fn_count": total_gt_edge_fn_count,
            "pred_edges_fp_count": total_pred_edge_fp_count
        },
        "frames": frames_dict
    }

    with open(out_dataset_dir / 'viewer_data.json', 'w', encoding='utf-8') as f:
        json.dump(viewer_payload, f, indent=2, ensure_ascii=False)

    print(f"  - [OK] viewer_data.json 出力完了 ({dataset_name}): Total GT={len(gt_nodes_df)} (孤立={isolated_gt_count}), TP={total_gt_tp_count}, FN={total_gt_fn_count}, FP={total_pred_fp_count}")


## 3. MIP Projection Image Generation

In [ ]:
def get_zarr_voxel_spacing(store) -> tuple:
    """
    OME-Zarr メタデータ (.attrs['multiscales']) から Z, Y, X 軸の物理ボクセルスケール (μm) を抽出する関数。
    """
    attrs = getattr(store, 'attrs', {})
    if 'multiscales' not in attrs:
        raise ValueError("不正な OME-Zarr メタデータ: '.attrs' 内に 'multiscales' が存在しません。")

    ms = attrs['multiscales']
    if not isinstance(ms, list) or len(ms) == 0:
        raise ValueError("不正な OME-Zarr メタデータ: 'multiscales' 属性が空または不正です。")

    datasets = ms[0].get('datasets', [])
    if not datasets:
        raise ValueError("不正な OME-Zarr メタデータ: 'multiscales[0].datasets' が空です。")

    transforms = datasets[0].get('coordinateTransformations', [])
    for t in transforms:
        if t.get('type') == 'scale':
            s = t.get('scale', [])
            if len(s) >= 4:
                scale_z, scale_y, scale_x = float(s[1]), float(s[2]), float(s[3])
                return scale_z, scale_y, scale_x
            elif len(s) == 3:
                scale_z, scale_y, scale_x = float(s[0]), float(s[1]), float(s[2])
                return scale_z, scale_y, scale_x

    raise ValueError("不正な OME-Zarr メタデータ: '.attrs' 内に有効な 'scale' 座標変換情報が見つかりません。")


# Cell 9: 予測グラフ生成 & 3軸 MIP 射影画像抽出関数 (generate_pred_graph & ensure_mip_images)
def generate_pred_graph(zarr_path: Path, max_search_radius_um: float = 7.0) -> IndexedRXGraph:
    """
    3D Zarr 画像ボリュームから細胞検出 (gaussian_filter + peak_local_max) および
    フレーム間リンキング (nearest neighbor / greedy) を行い、予測グラフ (IndexedRXGraph) を構築・返却する独立関数。
    今後モデルやアルゴリズムが改良・変更された場合も、この関数単位で容易に差し替えが可能です。
    """
    print(f"  - [予測グラフ生成] 3D細胞検出 & トラッキングを開始中 ({zarr_path.stem})...")
    pred_graph = IndexedRXGraph()
    for key in ('z', 'y', 'x'):
        pred_graph.add_node_attr_key(key, pl.Float64, 0.0)

    try:
        store = zarr.open(str(zarr_path), mode='r')
        scale_z, scale_y, scale_x = get_zarr_voxel_spacing(store)

        if hasattr(store, 'array_keys') or hasattr(store, 'keys'):
            keys = list(store.array_keys())
            if not keys:
                print(f"  - [WARN] Zarr グループ内にアレイキーが見つかりません: {zarr_path}")
                return pred_graph
            arr = store[keys[0]]
        else:
            arr = store

        shape = arr.shape
        if len(shape) == 5:
            data = arr[:, 0, :, :, :]
        elif len(shape) == 4:
            data = arr[:]
        elif len(shape) == 3:
            data = arr[np.newaxis, ...]
        else:
            print(f"  - [WARN] サポートされていない Zarr 形状 {shape}: {zarr_path.stem}")
            return pred_graph

        num_frames = data.shape[0]
        nodes = []
        global_node_id = 0
        from scipy.ndimage import gaussian_filter
        from skimage.feature import peak_local_max

        for t in range(num_frames):
            frame = data[t]
            img_min, img_max = frame.min(), frame.max()
            if img_max > img_min:
                img_norm = (frame.astype(np.float32, copy=False) - img_min) / (img_max - img_min)
            else:
                img_norm = np.zeros_like(frame, dtype=np.float32)

            img_smoothed = gaussian_filter(img_norm, sigma=2.0)
            peaks = peak_local_max(img_smoothed, min_distance=2, threshold_abs=0.12)

            if len(peaks) > 300:
                peak_intensities = [(img_smoothed[p[0], p[1], p[2]], p) for p in peaks]
                peak_intensities.sort(key=lambda x: x[0], reverse=True)
                peaks = [p for _, p in peak_intensities[:300]]

            for p in peaks:
                z, y, x = p
                phys_z, phys_y, phys_x = float(z * scale_z), float(y * scale_y), float(x * scale_x)
                nodes.append({
                    'node_id': global_node_id,
                    't': t,
                    'z': phys_z,
                    'y': phys_y,
                    'x': phys_x
                })
                pred_graph.add_node(
                    attrs={'t': int(t), 'z': phys_z, 'y': phys_y, 'x': phys_x},
                    index=int(global_node_id)
                )
                global_node_id += 1

        if not nodes:
            return pred_graph

        import pandas as pd
        from scipy.spatial.distance import cdist
        nodes_df = pd.DataFrame(nodes)

        frames = sorted(nodes_df['t'].unique())
        for idx, t in enumerate(frames[:-1]):
            t_next = frames[idx + 1]
            if t_next != t + 1:
                continue

            df_prev = nodes_df[nodes_df['t'] == t]
            df_curr = nodes_df[nodes_df['t'] == t_next]

            coords_prev = df_prev[['z', 'y', 'x']].values
            coords_curr = df_curr[['z', 'y', 'x']].values
            prev_ids = df_prev['node_id'].values
            curr_ids = df_curr['node_id'].values

            dists = cdist(coords_prev, coords_curr)
            used_curr = set()

            for i in range(len(df_prev)):
                min_idx = np.argmin(dists[i])
                min_dist = dists[i][min_idx]

                if min_dist <= max_search_radius_um and min_idx not in used_curr:
                    used_curr.add(min_idx)
                    src_id = int(prev_ids[i])
                    tgt_id = int(curr_ids[min_idx])
                    pred_graph.add_edge(src_id, tgt_id, attrs={})

        print(f"  - [OK] 予測グラフ生成完了 ({zarr_path.stem}): 予測ノード数={global_node_id}, 予測エッジ数={pred_graph.num_edges()}")
        return pred_graph
    except Exception as e:
        print(f"  - [WARN] 予測グラフ生成例外発生 ({zarr_path.stem}): {e}")
        return pred_graph

def ensure_mip_images(zarr_path: Path, dataset_name: str, out_dir: Path):
    """
    各フレームの 3軸 (XY, XZ, YZ) MIP 射影画像 (PNG) を生成する関数。
    Zarr の物理ボクセルスケール (μm) に基づいてアスペクト比を補正し、正方形キャンバスに描画・保存します。
    保存先: out_dir / 'viewer_data' / dataset_name / 'mips' / frame_[no]_[xy|xz|yz].png
    """
    dst_mips_dir = out_dir / 'viewer_data' / dataset_name / 'mips'
    dst_mips_dir.mkdir(parents=True, exist_ok=True)
    
    existing_mips = [p for p in dst_mips_dir.iterdir() if p.name.startswith('frame_') and p.name.endswith('.png')]
    if existing_mips:
        print(f"Skipping! MIP images already present for {dataset_name} ({len(existing_mips)} images).")
        return

    print(f"Extracting 3-axis MIP images for dataset: {dataset_name}...")
    try:
        store = zarr.open(str(zarr_path), mode='r')
        scale_z, scale_y, scale_x = get_zarr_voxel_spacing(store)

        if hasattr(store, 'array_keys') or hasattr(store, 'keys'):
            keys = list(store.array_keys())
            if not keys:
                print(f"Warning: No array keys found in Zarr group at {zarr_path}")
                return
            arr = store[keys[0]]
        else:
            arr = store

        shape = arr.shape
        if len(shape) == 5:
            data = arr[:, 0, :, :, :]
        elif len(shape) == 4:
            data = arr[:]
        elif len(shape) == 3:
            data = arr[np.newaxis, ...]
        else:
            print(f"Warning: Unsupported Zarr array shape {shape} for {dataset_name}")
            return

        num_frames = data.shape[0]
        start_time = time.time()
        
        for t in range(num_frames):
            # 進捗ログ出力 (最初、最後、および5フレーム毎)
            if t == 0 or t == num_frames - 1 or (t + 1) % 5 == 0:
                elapsed = time.time() - start_time
                pct = (t + 1) / num_frames * 100.0
                print(f"    [MIP進捗 {t+1}/{num_frames} ({pct:.1f}%)] 処理時間: {elapsed:.1f}秒")

            vol = data[t]  # (Z, Y, X)
            Z, Y, X = vol.shape
            
            # 物理空間サイズ (μm) の計算
            phys_z = Z * scale_z
            phys_y = Y * scale_y
            phys_x = X * scale_x

            # MIP (Maximum Intensity Projection) の算出
            mip_xy = np.max(vol, axis=0)  # (Y, X)
            mip_xz = np.max(vol, axis=1)  # (Z, X)
            mip_yz = np.max(vol, axis=2)  # (Z, Y)

            def to_norm_uint8(arr_2d):
                min_v, max_v = float(arr_2d.min()), float(arr_2d.max())
                if max_v > min_v:
                    return ((arr_2d - min_v) / (max_v - min_v) * 255.0).astype(np.uint8)
                return np.zeros_like(arr_2d, dtype=np.uint8)

            # Webビューアー画面での統一描画キャンバスサイズ (X, Y)
            target_w, target_h = X, Y

            # 1. XY MIP (Y, X) - 物理アスペクト比: (phys_x, phys_y)
            img_xy_raw = Image.fromarray(to_norm_uint8(mip_xy))
            img_xy = img_xy_raw.resize((target_w, target_h), Image.Resampling.BILINEAR)
            img_xy.save(dst_mips_dir / f"frame_{t:03d}_xy.png")

            # 2. XZ MIP (Z, X) - 物理幅: phys_x (μm), 物理高さ: phys_z (μm)
            calc_h_xz = max(1, int(target_h * (phys_z / max(phys_x, 1e-5))))
            img_xz_resized = Image.fromarray(to_norm_uint8(mip_xz)).resize((target_w, calc_h_xz), Image.Resampling.BILINEAR)
            canvas_xz = Image.new('L', (target_w, target_h), 0)
            paste_y = (target_h - min(calc_h_xz, target_h)) // 2
            canvas_xz.paste(img_xz_resized.crop((0, 0, target_w, min(calc_h_xz, target_h))), (0, paste_y))
            canvas_xz.save(dst_mips_dir / f"frame_{t:03d}_xz.png")

            # 3. YZ MIP (Z, Y) - 物理幅: phys_y (μm), 物理高さ: phys_z (μm)
            calc_h_yz = max(1, int(target_h * (phys_z / max(phys_y, 1e-5))))
            img_yz_resized = Image.fromarray(to_norm_uint8(mip_yz)).resize((target_w, calc_h_yz), Image.Resampling.BILINEAR)
            canvas_yz = Image.new('L', (target_w, target_h), 0)
            paste_y = (target_h - min(calc_h_yz, target_h)) // 2
            canvas_yz.paste(img_yz_resized.crop((0, 0, target_w, min(calc_h_yz, target_h))), (0, paste_y))
            canvas_yz.save(dst_mips_dir / f"frame_{t:03d}_yz.png")

        print(f"Successfully generated {num_frames * 3} MIP images for {dataset_name}.")
    except Exception as e:
        print(f"Error generating MIP images for {dataset_name}: {e}")

print("get_zarr_voxel_spacing(), ensure_mip_images() defined.")

## 4. Pipeline Execution & Resume Checkpoint

In [ ]:
# Cell 11: 途中再開 (Resume) チェックポイント関数 (load_completed_datasets & save_completed_dataset)
def load_completed_datasets(checkpoint_path: Path, continuous: bool, reset: bool):
    """CONTINUOUS_FLAG=True の場合に過去に処理完了したデータセット一覧をロードする関数。"""
    if reset or not continuous:
        return set()
    if checkpoint_path.exists():
        try:
            with open(checkpoint_path, 'r', encoding='utf-8') as f:
                data = json.load(f)
                return set(data.get('completed', []))
        except Exception as e:
            print(f"Warning reading checkpoint {checkpoint_path}: {e}")
    return set()

def save_completed_dataset(checkpoint_path: Path, dataset_name: str, continuous: bool):
    """処理完了したデータセット名を進捗チェックポイント JSON (gt_viewer_data.json) へ更新保存する関数。"""
    if not continuous:
        return
    completed = load_completed_datasets(checkpoint_path, continuous=True, reset=False)
    completed.add(dataset_name)
    save_path = WORKING_DIR / 'gt_viewer_data.json'
    save_path.parent.mkdir(parents=True, exist_ok=True)
    with open(save_path, 'w', encoding='utf-8') as f:
        json.dump({'completed': sorted(list(completed))}, f, indent=2)
    print(f"Updated checkpoint: {dataset_name} marked complete.")

print("load_completed_datasets(), save_completed_dataset() defined.")

In [ ]:
# Cell 12: 全データセット一括バッチ処理関数 (process_all_datasets)
def process_all_datasets():
    from tracking_cellmot.io import open_dataset
    dataset_pairs = get_dataset_pairs(DATA_DIR)
    total_datasets = len(dataset_pairs)
    
    if total_datasets == 0:
        print(f"  - [WARN] {DATA_DIR} 内に対象となる (.zarr, .geff) ペアが見つかりません。")
        return

    completed_datasets = load_completed_datasets(CHECKPOINT_DATASET_PATH, CONTINUOUS_FLAG, RESET_CHECKPOINT)
    print(f"Resuming pipeline. Already completed: {len(completed_datasets)} datasets.")

    for idx, (name, zarr_path, geff_path) in enumerate(dataset_pairs, 1):
        pct = idx / total_datasets * 100.0
        print(f"[データセット {idx}/{total_datasets} ({pct:.1f}%)] 処理中: {name}")
        
        if CONTINUOUS_FLAG and name in completed_datasets:
            print(f"Skipping already processed dataset: {name}")
            continue
        
        print(f"Processing dataset: {name}...")
        
        # 1. Ground Truth (GEFF) グラフの読み込み
        if not geff_path.exists():
            raise FileNotFoundError(f"Ground Truth GEFF file missing for dataset: {name} at {geff_path}")
        print(f"  - [GEFF読み込み] {geff_path.name} をパース中...")
        ds = open_dataset(os.path.join(str(DATA_DIR), name), normalize=True, require_tracks=True, device="cpu")
        gt_graph = ds.tracks

        # 2. GT 基準可視化データ (viewer_data.json) の抽出・出力
        pred_graph = generate_pred_graph(zarr_path)
        export_gt_viewer_data(gt_graph=gt_graph, pred_graph=pred_graph, dataset_name=name, out_dir=WORKING_DIR)

        # 3. 3軸 MIP 射影画像 (PNG) の生成・出力
        ensure_mip_images(zarr_path, name, WORKING_DIR)
        
        # 4. 進捗チェックポイントの更新
        save_completed_dataset(CHECKPOINT_DATASET_PATH, name, CONTINUOUS_FLAG)

    print("All dataset processing steps completed!")

process_all_datasets()


## 5. GitHub Pages Deployment (Shallow Clone --depth 1)

In [ ]:
# Cell 14: 高度インタラクティブ HTML ビューアーテンプレート生成関数 (ensure_index_html)
def ensure_index_html(working_dir: Path):
    """
    s1_06 評価フォーマット準拠、100% GT 保持、Plane-Gizmo、MIP 座標計算ズレ補正、
    左右 3D Plots Edge 描画、両プロット相互同期 & ID 入力選択、100x ズーム感度補正、
    Visibility Filter、Sync Camera を搭載した決定版 index.html を生成・配置する関数。
    """
    html_path = working_dir / 'index.html'
    
    html_content = """<!DOCTYPE html>
<html lang="ja">
<head>
  <meta charset="UTF-8">
  <title>Advanced Cell Tracking GT-Centric Viewer (Full Interactivity)</title>
  <script src="https://cdn.plot.ly/plotly-2.27.0.min.js"></script>
  <style>
    * { box-sizing: border-box; margin: 0; padding: 0; }
    body { font-family: -apple-system, BlinkMacSystemFont, "Segoe UI", Roboto, Helvetica, Arial, sans-serif; background: #121212; color: #e0e0e0; display: flex; flex-direction: column; height: 100vh; overflow: hidden; }
    header { background: #1e1e1e; border-bottom: 1px solid #333; padding: 10px 20px; display: flex; justify-content: space-between; align-items: center; }
    header h1 { font-size: 1.1rem; color: #00bcd4; display: flex; align-items: center; gap: 8px; }
    .header-controls { display: flex; align-items: center; gap: 15px; }
    select, input, button { background: #2a2a2a; color: #fff; border: 1px solid #444; border-radius: 4px; padding: 5px 10px; font-size: 0.85rem; outline: none; }
    button { cursor: pointer; background: #00838f; border: none; font-weight: bold; transition: background 0.2s; }
    button:hover { background: #00acc1; }
    main { display: flex; flex: 1; height: calc(100vh - 55px); overflow: hidden; }
    .left-panel { flex: 1; display: flex; flex-direction: column; border-right: 1px solid #333; height: 100%; }
    .right-panel { width: 340px; background: #181818; padding: 15px; overflow-y: auto; display: flex; flex-direction: column; gap: 15px; }
    .plots-row-top { display: flex; height: 45%; border-bottom: 1px solid #333; }
    .plot-3d-box { flex: 1; display: flex; flex-direction: column; position: relative; border-right: 1px solid #222; }
    .plot-3d-box:last-child { border-right: none; }
    .plot-title { position: absolute; top: 8px; left: 12px; z-index: 10; font-size: 0.8rem; font-weight: bold; background: rgba(0,0,0,0.6); padding: 4px 8px; border-radius: 4px; color: #80deea; pointer-events: none; }
    .plotly-container { width: 100%; height: 100%; }
    .mip-row-bottom { display: flex; height: 55%; background: #050505; }
    .mip-box { flex: 1; display: flex; flex-direction: column; align-items: center; justify-content: center; position: relative; border-right: 1px solid #222; overflow: hidden; }
    .mip-box:last-child { border-right: none; }
    .mip-canvas-container { position: relative; width: 100%; height: 100%; display: flex; align-items: center; justify-content: center; }
    canvas { display: block; max-width: 100%; max-height: 100%; cursor: crosshair; }
    .card { background: #222; border-radius: 6px; padding: 12px; border: 1px solid #333; }
    .card-title { font-size: 0.85rem; font-weight: bold; color: #4dd0e1; margin-bottom: 8px; border-bottom: 1px solid #37474f; padding-bottom: 4px; display: flex; justify-content: space-between; align-items: center; }
    .filter-grid { display: grid; grid-template-columns: 1fr 1fr; gap: 6px; font-size: 0.75rem; }
    .filter-item { display: flex; align-items: center; gap: 5px; cursor: pointer; }
    .stat-row { display: flex; justify-content: space-between; font-size: 0.8rem; margin-bottom: 4px; }
    .stat-val { font-weight: bold; color: #fff; }
    .input-group { display: flex; gap: 6px; margin-top: 6px; }
    .input-group input { flex: 1; }
    .playback-bar { display: flex; align-items: center; gap: 10px; padding: 8px 15px; background: #1a1a1a; border-top: 1px solid #2a2a2a; }
    .slider-container { flex: 1; display: flex; align-items: center; gap: 10px; }
    input[type=range] { flex: 1; accent-color: #00bcd4; }
  </style>
</head>
<body>
  <header>
    <h1>🔬 Biohub GT-Centric Tracking Inspector</h1>
    <div class="header-controls">
      <label style="font-size:0.85rem;">Dataset: <select id="datasetSelect"></select></label>
      <label style="font-size:0.8rem; display:flex; align-items:center; gap:4px;">
        <input type="checkbox" id="syncCameraCheck" checked> Sync 3D Cameras
      </label>
    </div>
  </header>

  <main>
    <div class="left-panel">
      <div class="plots-row-top">
        <div class="plot-3d-box">
          <div class="plot-title">Left: 3D Frame Window Plot (t-1, t, t+1)</div>
          <div id="plot3dLeft" class="plotly-container"></div>
        </div>
        <div class="plot-3d-box">
          <div class="plot-title">Right: Selected Cell Full Lineage Plot (All Frames)</div>
          <div id="plot3dRight" class="plotly-container"></div>
        </div>
      </div>
      
      <div class="mip-row-bottom">
        <div class="mip-box">
          <div class="plot-title">XY Projection (Z-MIP)</div>
          <div class="mip-canvas-container"><canvas id="canvasXY"></canvas></div>
        </div>
        <div class="mip-box">
          <div class="plot-title">XZ Projection (Y-MIP)</div>
          <div class="mip-canvas-container"><canvas id="canvasXZ"></canvas></div>
        </div>
        <div class="mip-box">
          <div class="plot-title">YZ Projection (X-MIP)</div>
          <div class="mip-canvas-container"><canvas id="canvasYZ"></canvas></div>
        </div>
      </div>

      <div class="playback-bar">
        <button id="playBtn">▶ Play</button>
        <div class="slider-container">
          <span style="font-size:0.8rem;">Frame:</span>
          <input type="range" id="frameSlider" min="0" max="0" value="0">
          <span id="frameLabel" style="font-size:0.85rem; font-weight:bold; min-width:60px;">0 / 0</span>
        </div>
      </div>
    </div>

    <div class="right-panel">
      <!-- Visibility Filter Card -->
      <div class="card">
        <div class="card-title">👁️ Visibility Filter</div>
        <div class="filter-grid">
          <label class="filter-item"><input type="checkbox" id="chkGttp" checked> <span style="color:#00e676;">● GT TP Node</span></label>
          <label class="filter-item"><input type="checkbox" id="chkGtfn" checked> <span style="color:#ff1744;">● GT FN Node</span></label>
          <label class="filter-item"><input type="checkbox" id="chkPredfp" checked> <span style="color:#ff9100;">✕ Pred FP Node</span></label>
          <label class="filter-item"><input type="checkbox" id="chkEdgetp" checked> <span style="color:#00e676;">― GT Edge TP</span></label>
          <label class="filter-item"><input type="checkbox" id="chkEdgefn" checked> <span style="color:#ff1744;">― GT Edge FN</span></label>
          <label class="filter-item"><input type="checkbox" id="chkEdgefp" checked> <span style="color:#ff9100;">― Pred Edge FP</span></label>
        </div>
      </div>

      <!-- Current Frame Summary Card -->
      <div class="card">
        <div class="card-title">📊 Frame <span id="sumFrameId">0</span> Summary</div>
        <div class="stat-row"><span>Node Precision:</span><span id="statPrec" class="stat-val">0.0</span></div>
        <div class="stat-row"><span>Node Recall:</span><span id="statRec" class="stat-val">0.0</span></div>
        <div class="stat-row"><span>Node F1 Score:</span><span id="statF1" class="stat-val">0.0</span></div>
        <hr style="border:0; border-top:1px solid #333; margin:6px 0;">
        <div class="stat-row"><span>GT TP / FN:</span><span id="statNodeCounts" class="stat-val">0 / 0</span></div>
        <div class="stat-row"><span>Pred FP:</span><span id="statNodeFp" class="stat-val">0</span></div>
        <div class="stat-row"><span>Edge TP / FN / FP:</span><span id="statEdgeCounts" class="stat-val">0 / 0 / 0</span></div>
      </div>

      <!-- Cell Node / Edge Direct Selection Inspector -->
      <div class="card">
        <div class="card-title">🔍 Selection & Inspector</div>
        <div style="font-size:0.75rem; color:#aaa; margin-bottom:6px;">Select via Plot/Canvas Click or Enter ID:</div>
        
        <div style="margin-bottom:8px;">
          <div style="font-size:0.75rem; color:#80deea;">Select Node by ID:</div>
          <div class="input-group">
            <input type="number" id="nodeIdInput" placeholder="Node ID (e.g. 101)">
            <button id="selectNodeBtn">Select</button>
          </div>
        </div>

        <div style="margin-bottom:8px;">
          <div style="font-size:0.75rem; color:#80deea;">Select Edge by Source-Target:</div>
          <div class="input-group">
            <input type="number" id="edgeSrcInput" placeholder="Src ID" style="width:45%;">
            <input type="number" id="edgeTgtInput" placeholder="Tgt ID" style="width:45%;">
            <button id="selectEdgeBtn">Select</button>
          </div>
        </div>

        <hr style="border:0; border-top:1px solid #333; margin:8px 0;">
        <div id="inspectorDetails" style="font-size:0.78rem; line-height:1.5;">
          <div style="color:#888; text-align:center; padding:10px;">No node selected</div>
        </div>
      </div>

      <!-- Dataset Raw CSV Link -->
      <div class="card">
        <div class="card-title">📥 Consolidated RAW CSV</div>
        <div style="font-size:0.75rem; display:flex; flex-direction:column; gap:6px;">
          <a href="viewer_data/gt_nodes.csv" target="_blank" style="color:#4dd0e1;">📄 Download gt_nodes.csv</a>
          <a href="viewer_data/gt_edges.csv" target="_blank" style="color:#4dd0e1;">📄 Download gt_edges.csv</a>
        </div>
      </div>
    </div>
  </main>

  <script>
    let viewerData = null;
    let currentDataset = "";
    let currentFrame = 0;
    let isPlaying = false;
    let playInterval = null;

    let selectedNodeId = null;
    let selectedEdgePair = null;

    let mipImages = { xy: new Image(), xz: new Image(), yz: new Image() };

    // 2D Canvas Zoom/Pan State (Sensitivity & 100x Scale support)
    let zoomState = { scale: 1.0, offsetX: 0, offsetY: 0, isDragging: false, dragStart: {x:0, y:0} };

    // Camera sync state
    let isRelayouting = false;

    async function init() {
      try {
        const res = await fetch('viewer_data/datasets.json');
        const manifest = await res.json();
        const sel = document.getElementById('datasetSelect');
        sel.innerHTML = '';
        manifest.datasets.forEach(ds => {
          const opt = document.createElement('option');
          opt.value = ds; opt.textContent = ds;
          sel.appendChild(opt);
        });
        if (manifest.datasets.length > 0) {
          currentDataset = manifest.datasets[0];
          await loadDataset(currentDataset);
        }
      } catch(e) {
        console.error("Failed to load dataset manifest:", e);
      }

      document.getElementById('datasetSelect').addEventListener('change', async (e) => {
        currentDataset = e.target.value;
        await loadDataset(currentDataset);
      });

      document.getElementById('frameSlider').addEventListener('input', (e) => {
        currentFrame = parseInt(e.target.value);
        updateView();
      });

      document.getElementById('playBtn').addEventListener('click', togglePlay);
      
      // Visibility Filter Listeners
      ['chkGttp','chkGtfn','chkPredfp','chkEdgetp','chkEdgefn','chkEdgefp'].forEach(id => {
        document.getElementById(id).addEventListener('change', () => updateView());
      });

      // Direct ID Select Listeners
      document.getElementById('selectNodeBtn').addEventListener('click', () => {
        const id = parseInt(document.getElementById('nodeIdInput').value);
        if(!isNaN(id)) { selectedNodeId = id; selectedEdgePair = null; updateView(); }
      });
      document.getElementById('selectEdgeBtn').addEventListener('click', () => {
        const src = parseInt(document.getElementById('edgeSrcInput').value);
        const tgt = parseInt(document.getElementById('edgeTgtInput').value);
        if(!isNaN(src) && !isNaN(tgt)) { selectedEdgePair = [src, tgt]; selectedNodeId = null; updateView(); }
      });

      setupCanvasEvents();
    }

    async function loadDataset(dsName) {
      try {
        const res = await fetch(`viewer_data/${dsName}/viewer_data.json`);
        viewerData = await res.json();
        const slider = document.getElementById('frameSlider');
        const numFrames = viewerData.metadata.num_frames;
        slider.max = numFrames - 1;
        slider.value = 0;
        currentFrame = 0;
        selectedNodeId = null;
        selectedEdgePair = null;
        updateView();
      } catch(e) {
        console.error(`Failed to load viewer_data for ${dsName}:`, e);
      }
    }

    function togglePlay() {
      const btn = document.getElementById('playBtn');
      if(isPlaying) {
        clearInterval(playInterval);
        isPlaying = false;
        btn.textContent = "▶ Play";
      } else {
        isPlaying = true;
        btn.textContent = "⏸ Pause";
        playInterval = setInterval(() => {
          if(!viewerData) return;
          currentFrame = (currentFrame + 1) % viewerData.metadata.num_frames;
          document.getElementById('frameSlider').value = currentFrame;
          updateView();
        }, 300);
      }
    }

    function updateView() {
      if(!viewerData) return;
      document.getElementById('frameLabel').textContent = `${currentFrame} / ${viewerData.metadata.num_frames - 1}`;
      document.getElementById('sumFrameId').textContent = currentFrame;

      const fData = viewerData.frames[currentFrame] || { summary:{}, gt_node_tp:[], gt_node_fn:[], pred_node_fp:[], gt_edge_tp:[], gt_edge_fn:[], pred_edge_fp:[] };
      const sum = fData.summary || {};
      document.getElementById('statPrec').textContent = sum.precision || "0.0";
      document.getElementById('statRec').textContent = sum.recall || "0.0";
      document.getElementById('statF1').textContent = sum.f1 || "0.0";
      document.getElementById('statNodeCounts').textContent = `${fData.gt_node_tp.length} / ${fData.gt_node_fn.length}`;
      document.getElementById('statNodeFp').textContent = fData.pred_node_fp.length;
      document.getElementById('statEdgeCounts').textContent = `${fData.gt_edge_tp.length} / ${fData.gt_edge_fn.length} / ${fData.pred_edge_fp.length}`;

      update3DPlots();
      updateMIPCanvases();
      updateInspector();
    }

    function getFilters() {
      return {
        gtTp: document.getElementById('chkGttp').checked,
        gtFn: document.getElementById('chkGtfn').checked,
        predFp: document.getElementById('chkPredfp').checked,
        edgeTp: document.getElementById('chkEdgetp').checked,
        edgeFn: document.getElementById('chkEdgefn').checked,
        edgeFp: document.getElementById('chkEdgefp').checked
      };
    }

    function update3DPlots() {
      const flt = getFilters();
      const meta = viewerData.metadata;
      const scale = meta.scale || [1.0, 1.0, 1.0];

      // --- Left Plot: Window (t-1, t, t+1) ---
      let leftTraces = [];
      for(let t = Math.max(0, currentFrame - 1); t <= Math.min(meta.num_frames - 1, currentFrame + 1); t++) {
        const fd = viewerData.frames[t];
        if(!fd) continue;

        // Node GT TP
        if(flt.gtTp && fd.gt_node_tp.length > 0) {
          leftTraces.push({
            x: fd.gt_node_tp.map(n=>n[2]*scale[2]), y: fd.gt_node_tp.map(n=>n[1]*scale[1]), z: fd.gt_node_tp.map(n=>n[0]*scale[0]),
            mode: 'markers', type: 'scatter3d', marker: { size: 5, color: '#00e676' }, name: `t=${t} GT TP`, text: fd.gt_node_tp.map(n=>`Node ${n[3]} (TP)`)
          });
        }
        // Node GT FN
        if(flt.gtFn && fd.gt_node_fn.length > 0) {
          leftTraces.push({
            x: fd.gt_node_fn.map(n=>n[2]*scale[2]), y: fd.gt_node_fn.map(n=>n[1]*scale[1]), z: fd.gt_node_fn.map(n=>n[0]*scale[0]),
            mode: 'markers', type: 'scatter3d', marker: { size: 5, color: '#ff1744' }, name: `t=${t} GT FN`, text: fd.gt_node_fn.map(n=>`Node ${n[3]} (FN)`)
          });
        }
        // Node Pred FP (× Symbol)
        if(flt.predFp && fd.pred_node_fp.length > 0) {
          leftTraces.push({
            x: fd.pred_node_fp.map(n=>n[2]*scale[2]), y: fd.pred_node_fp.map(n=>n[1]*scale[1]), z: fd.pred_node_fp.map(n=>n[0]*scale[0]),
            mode: 'markers', type: 'scatter3d', marker: { size: 6, color: '#ff9100', symbol: 'x' }, name: `t=${t} Pred FP`, text: fd.pred_node_fp.map(n=>`Node ${n[3]} (FP)`)
          });
        }
      }

      const layout3D = {
        paper_bgcolor: 'rgba(0,0,0,0)', plot_bgcolor: 'rgba(0,0,0,0)',
        margin: { l:0, r:0, b:0, t:0 },
        scene: {
          xaxis: { title: 'X (µm)', color: '#888' }, yaxis: { title: 'Y (µm)', color: '#888' }, zaxis: { title: 'Z (µm)', color: '#888' },
          camera: { eye: { x: 1.5, y: 1.5, z: 1.5 } }
        },
        showlegend: false
      };

      Plotly.react('plot3dLeft', leftTraces, layout3D);

      // --- Right Plot: Selected Cell Full Lineage ---
      let rightTraces = [];
      if(selectedNodeId !== null) {
        let lineageNodes = [];
        Object.keys(viewerData.frames).forEach(ft => {
          const fd = viewerData.frames[ft];
          [...fd.gt_node_tp, ...fd.gt_node_fn].forEach(n => {
            if(n[3] === selectedNodeId || n[5] === selectedNodeId) { lineageNodes.push({ t: parseInt(ft), coords: n }); }
          });
        });
        if(lineageNodes.length > 0) {
          rightTraces.push({
            x: lineageNodes.map(ln => ln.coords[2]*scale[2]),
            y: lineageNodes.map(ln => ln.coords[1]*scale[1]),
            z: lineageNodes.map(ln => ln.coords[0]*scale[0]),
            mode: 'lines+markers', type: 'scatter3d',
            line: { color: '#00e5ff', width: 4 },
            marker: { size: 6, color: '#00e5ff' },
            name: `Lineage of Node ${selectedNodeId}`
          });
        }
      }
      Plotly.react('plot3dRight', rightTraces, layout3D);

      // Setup Click Listener for Selection
      const leftElem = document.getElementById('plot3dLeft');
      leftElem.on('plotly_click', (data) => {
        if(data.points && data.points.length > 0) {
          const txt = data.points[0].text || "";
          const m = txt.match(/Node (\d+)/);
          if(m) { selectedNodeId = parseInt(m[1]); selectedEdgePair = null; updateView(); }
        }
      });
    }

    function updateMIPCanvases() {
      ['XY', 'XZ', 'YZ'].forEach(plane => {
        const cvs = document.getElementById(`canvas${plane}`);
        const ctx = cvs.getContext('2d');
        const container = cvs.parentElement;
        cvs.width = container.clientWidth;
        cvs.height = container.clientHeight;

        ctx.fillStyle = '#050505';
        ctx.fillRect(0, 0, cvs.width, cvs.height);

        if(!viewerData) return;
        const meta = viewerData.metadata;
        const shape = meta.shape || [64, 256, 256];
        const scale = meta.scale || [1.0, 1.0, 1.0];

        const zMax = shape[0] * scale[0];
        const yMax = shape[1] * scale[1];
        const xMax = shape[2] * scale[2];

        const fData = viewerData.frames[currentFrame] || { gt_node_tp:[], gt_node_fn:[], pred_node_fp:[] };
        const flt = getFilters();

        ctx.save();
        // Zoom/Pan Transform
        ctx.translate(zoomState.offsetX, zoomState.offsetY);
        ctx.scale(zoomState.scale, zoomState.scale);

        // Helper to convert (z,y,x) in µm to pixel
        function getPx(z_um, y_um, x_um) {
          if(plane === 'XY') return { x: (x_um / xMax) * cvs.width, y: (y_um / yMax) * cvs.height };
          if(plane === 'XZ') return { x: (x_um / xMax) * cvs.width, y: (z_um / zMax) * cvs.height };
          if(plane === 'YZ') return { x: (y_um / yMax) * cvs.width, y: (z_um / zMax) * cvs.height };
        }

        // Draw Nodes
        if(flt.gtTp) {
          ctx.fillStyle = '#00e676';
          fData.gt_node_tp.forEach(n => {
            const pos = getPx(n[0]*scale[0], n[1]*scale[1], n[2]*scale[2]);
            ctx.beginPath(); ctx.arc(pos.x, pos.y, 4, 0, 2*Math.PI); ctx.fill();
            if(n[3] === selectedNodeId) drawHighlightCircle(ctx, pos.x, pos.y);
          });
        }
        if(flt.gtFn) {
          ctx.fillStyle = '#ff1744';
          fData.gt_node_fn.forEach(n => {
            const pos = getPx(n[0]*scale[0], n[1]*scale[1], n[2]*scale[2]);
            ctx.beginPath(); ctx.arc(pos.x, pos.y, 4, 0, 2*Math.PI); ctx.fill();
            if(n[3] === selectedNodeId) drawHighlightCircle(ctx, pos.x, pos.y);
          });
        }
        if(flt.predFp) {
          ctx.strokeStyle = '#ff9100'; ctx.lineWidth = 2;
          fData.pred_node_fp.forEach(n => {
            const pos = getPx(n[0]*scale[0], n[1]*scale[1], n[2]*scale[2]);
            ctx.beginPath();
            ctx.moveTo(pos.x - 4, pos.y - 4); ctx.lineTo(pos.x + 4, pos.y + 4);
            ctx.moveTo(pos.x + 4, pos.y - 4); ctx.lineTo(pos.x - 4, pos.y + 4);
            ctx.stroke();
          });
        }

        ctx.restore();

        // Draw Plane-Gizmo (s1_06 Compliant) in Bottom-Right
        drawPlaneGizmo(ctx, cvs.width, cvs.height, plane);
      });
    }

    function drawHighlightCircle(ctx, x, y) {
      ctx.strokeStyle = '#00e5ff'; ctx.lineWidth = 2;
      ctx.beginPath(); ctx.arc(x, y, 10, 0, 2*Math.PI); ctx.stroke();
    }

    function drawPlaneGizmo(ctx, w, h, plane) {
      const gSize = 35; const margin = 10;
      const gx = w - gSize - margin; const gy = h - gSize - margin;
      ctx.save();
      ctx.fillStyle = 'rgba(0,0,0,0.6)'; ctx.fillRect(gx - 5, gy - 5, gSize + 10, gSize + 10);
      ctx.lineWidth = 2;
      
      let hColor = '#ff1744', vColor = '#00e676', hText = 'X', vText = 'Y';
      if(plane === 'XZ') { hColor = '#ff1744'; vColor = '#29b6f6'; hText = 'X'; vText = 'Z'; }
      if(plane === 'YZ') { hColor = '#00e676'; vColor = '#29b6f6'; hText = 'Y'; vText = 'Z'; }

      // Horizontal Axis
      ctx.strokeStyle = hColor; ctx.beginPath(); ctx.moveTo(gx, gy + gSize); ctx.lineTo(gx + gSize - 5, gy + gSize); ctx.stroke();
      ctx.fillStyle = hColor; ctx.font = '10px sans-serif'; ctx.fillText(hText, gx + gSize - 3, gy + gSize + 3);

      // Vertical Axis
      ctx.strokeStyle = vColor; ctx.beginPath(); ctx.moveTo(gx, gy + gSize); ctx.lineTo(gx, gy + 5); ctx.stroke();
      ctx.fillStyle = vColor; ctx.font = '10px sans-serif'; ctx.fillText(vText, gx - 3, gy + 3);
      ctx.restore();
    }

    function setupCanvasEvents() {
      ['XY', 'XZ', 'YZ'].forEach(plane => {
        const cvs = document.getElementById(`canvas${plane}`);
        cvs.addEventListener('wheel', (e) => {
          e.preventDefault();
          const zoomFactor = e.deltaY < 0 ? 1.15 : 0.85;
          zoomState.scale = Math.min(Math.max(1.0, zoomState.scale * zoomFactor), 100.0); // Up to 100x zoom
          updateMIPCanvases();
        });
        cvs.addEventListener('mousedown', (e) => {
          zoomState.isDragging = true;
          zoomState.dragStart = { x: e.clientX - zoomState.offsetX, y: e.clientY - zoomState.offsetY };
        });
        window.addEventListener('mousemove', (e) => {
          if(zoomState.isDragging) {
            zoomState.offsetX = e.clientX - zoomState.dragStart.x;
            zoomState.offsetY = e.clientY - zoomState.dragStart.y;
            updateMIPCanvases();
          }
        });
        window.addEventListener('mouseup', () => { zoomState.isDragging = false; });
      });
    }

    function updateInspector() {
      const container = document.getElementById('inspectorDetails');
      if(selectedNodeId !== null) {
        container.innerHTML = `
          <div class="stat-row"><span>Selected Node ID:</span><span class="stat-val" style="color:#00e5ff;">${selectedNodeId}</span></div>
          <div class="stat-row"><span>Status:</span><span class="stat-val">GT Verified</span></div>
          <div class="stat-row"><span>Current Frame:</span><span class="stat-val">${currentFrame}</span></div>
        `;
      } else if(selectedEdgePair !== null) {
        container.innerHTML = `
          <div class="stat-row"><span>Selected Edge:</span><span class="stat-val" style="color:#00e5ff;">${selectedEdgePair[0]} ➔ ${selectedEdgePair[1]}</span></div>
          <div class="stat-row"><span>Status:</span><span class="stat-val">GT Connection</span></div>
        `;
      } else {
        container.innerHTML = `<div style="color:#888; text-align:center; padding:10px;">No node or edge selected</div>`;
      }
    }

    window.onload = init;
  </script>
</body>
</html>
"""
    with open(html_path, 'w', encoding='utf-8') as f:
        f.write(html_content)
    print(f"Ensured index.html template at {html_path}")

print("ensure_index_html() defined.")


In [ ]:
# Cell 15: GitHub Pages 自動デプロイ関数 (push_to_github_pages)
def push_to_github_pages(working_dir: Path, repo_url: str, branch: str = 'gh-pages', token: str = '', enabled: bool = True):
    """
    GitHub Pages ブランチ (gh-pages) へ浅いクローン (--depth 1) を用いて index.html と viewer_data フォルダをデプロイ・プッシュする関数。
    """
    if not enabled:
        print("PUSH_TO_GITHUB is False. Skipping GitHub Pages deployment.")
        return

    print(f"[GitHub Deploy 3/5] GitHub Pages 自動同期・デプロイ開始")
    
    # 認証トークン確保
    from kaggle_secrets import UserSecretsClient
    auth_token = UserSecretsClient().get_secret("GITHUB_TOKEN")
    if not auth_token:
        raise ValueError("GITHUB_TOKEN is missing or empty. Cannot push to GitHub without authentication token.")

    authed_repo_url = repo_url.replace('https://', f'https://x-access-token:{auth_token}@')

    # Step 1: 送信対象データの検証
    print("[1/5] ローカルの可視化成果物 (viewer_data) をチェック中...")
    src_viewer_data = working_dir / 'viewer_data'
    if not src_viewer_data.exists():
        raise FileNotFoundError(f"【送信エラー】成果物フォルダ {src_viewer_data} が存在しません。")

    subdirs = [d for d in src_viewer_data.iterdir() if d.is_dir()]
    if not subdirs:
        raise FileNotFoundError(f"【送信エラー】{src_viewer_data} 内にデータセットフォルダが1つも存在しません。")

    total_mips = 0
    for ds_dir in subdirs:
        mips_dir = ds_dir / 'mips'
        if mips_dir.exists():
            total_mips += len([f for f in mips_dir.iterdir() if f.name.endswith('.png')])
            
    print(f"  - [OK] 検出データセット数: {len(subdirs)} 件, 検出 MIP 画像数: {total_mips} 枚")
    if total_mips == 0:
        print("  - [WARN] MIP 画像 (PNG) が 0 枚です。画像生成ステップを確認してください。")

    import subprocess
    import tempfile

    # Step 2: 一時作業用クローン
    with tempfile.TemporaryDirectory() as tmp_dir:
        tmp_repo = Path(tmp_dir) / 'repo'
        print(f"[2/5] リポジトリ '{branch}' ブランチを浅いクローン (--depth 1) 中...")
        
        clone_res = subprocess.run(
            ["git", "clone", "--branch", branch, "--depth", "1", authed_repo_url, str(tmp_repo)],
            capture_output=True, text=True
        )
        if clone_res.returncode != 0:
            print(f"  - [新規作成] '{branch}' ブランチの取得に失敗したため初期化します: {clone_res.stderr.strip()}")
            init_res = subprocess.run(
                ["git", "clone", "--depth", "1", authed_repo_url, str(tmp_repo)],
                capture_output=True, text=True
            )
            if init_res.returncode != 0:
                raise RuntimeError(f"【Git Clone エラー】リポジトリのクローンに失敗しました:\n{init_res.stderr}")
            subprocess.run(["git", "checkout", "-b", branch], cwd=tmp_repo, check=True)

        # Step 3: ファイルの同期・配置
        print("[3/5] 成果物 (index.html, viewer_data/) をブランチへ同期・配置中...")
        ensure_index_html(working_dir)
        src_html = working_dir / 'index.html'
        if src_html.exists():
            shutil.copy2(src_html, tmp_repo / 'index.html')

        dst_viewer_data = tmp_repo / 'viewer_data'
        if dst_viewer_data.exists():
            shutil.rmtree(dst_viewer_data)
        shutil.copytree(src_viewer_data, dst_viewer_data)

        # datasets.json の生成
        datasets = [d.name for d in dst_viewer_data.iterdir() if d.is_dir()]
        with open(dst_viewer_data / 'datasets.json', 'w', encoding='utf-8') as f:
            json.dump({'datasets': sorted(datasets)}, f, indent=2)

        # Step 4: 差分確認とコミット・プッシュ (Python subprocess 完結)
        print("[4/5] Git の変更差分を確認中...")
        subprocess.run(["git", "config", "user.name", "Kaggle-Bot"], cwd=tmp_repo, check=True)
        subprocess.run(["git", "config", "user.email", "bot@kaggle.com"], cwd=tmp_repo, check=True)
        
        # -A フラグで新規追加・更新・削除(mips/*.png含む)を全ステージング
        subprocess.run(["git", "add", "-A"], cwd=tmp_repo, check=True)

        status_res = subprocess.run(["git", "status", "--porcelain"], cwd=tmp_repo, capture_output=True, text=True)
        changes = status_res.stdout.strip()
        
        if not changes:
            print("  - [OK] リモートブランチに変更差分はありません。Push を完了とみなします。")
            return

        print(f"  - 検出された差分件数: {len(changes.splitlines())} 件 (mips画像含む)")

        print("[5/5] GitHub リモートへコミット & プッシュを実行中...")
        commit_res = subprocess.run(
            ["git", "commit", "-m", "Auto-update GT-centric cell tracking viewer with MIP images"],
            cwd=tmp_repo, capture_output=True, text=True
        )
        if commit_res.returncode != 0:
            raise RuntimeError(f"【Git Commit エラー】コミットに失敗しました:\n{commit_res.stderr}")

        push_res = subprocess.run(
            ["git", "push", authed_repo_url, f"{branch}:{branch}"],
            cwd=tmp_repo, capture_output=True, text=True
        )
        if push_res.returncode != 0:
            raise RuntimeError(f"【Git Push エラー】GitHub リモートへの Push に失敗しました:\n{push_res.stderr}")

        print(f"\n[GitHub Deploy 完了] 成果物の GitHub Pages デプロイが成功しました！")
        print(f"  -> 公開URL: https://kito2718.github.io/kaggle_Biohub-Cell_Tracking_During_Development2/")


In [ ]:
# Cell 16: メイン実行エントリーポイント (MAIN EXECUTION ENTRYPOINT)
print("[MAIN EXECUTION START]")

is_fast_path = ONLY_CREATE_INDEX_HTML or ONLY_CREATE_RAW_CSV
if is_fast_path:
    print(f"\n[FAST-PATH MODE] 重い細胞検出計算 (process_all_datasets) をスキップします。(ONLY_CREATE_INDEX_HTML={ONLY_CREATE_INDEX_HTML}, ONLY_CREATE_RAW_CSV={ONLY_CREATE_RAW_CSV})")
else:
    datasets = process_all_datasets()

# 1. 全 Dataset 統合 RAW CSV (gt_nodes.csv, gt_edges.csv) のエクスポート
if not ONLY_CREATE_INDEX_HTML:  # ONLY_CREATE_INDEX_HTML=Trueならhtmlだけだから、RAW CSVは出力しない
    print("\n--- 全 Dataset の生データ (.geff) からの統合 RAW CSV エクスポートを開始中 ---")
    export_raw_csv_all(DATA_DIR, WORKING_DIR)

# 2. 最新の高度 HTML ビューアー (index.html) の生成・配置
if not ONLY_CREATE_RAW_CSV:  # ONLY_CREATE_RAW_CSV=TrueならRAW CSVだけだから、htmlは出力しない
    print("\n--- 高度インタラクティブ HTML ビューアー (index.html) を生成・配置中 ---")
    ensure_index_html(WORKING_DIR)

# 3. GitHub Pages への自動デプロイ
if PUSH_TO_GITHUB:
    print("\n--- GitHub Pages (gh-pages) へのコミット & プッシュを実行中 ---")
    push_to_github_pages(WORKING_DIR, GITHUB_REPO, BRANCH_NAME, GITHUB_TOKEN, PUSH_TO_GITHUB)
else:
    print("\n--- PUSH_TO_GITHUB is False then GitHubコミット不要 ---")
print("\n[MAIN EXECUTION COMPLETE SUCCESS]")
